In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuración de visualización
%matplotlib inline
sns.set_theme(style="whitegrid")

# Definir la ruta del archivo
# Usamos el archivo .pkl para mantener la integridad de los tipos de datos (especialmente los IDs de estados)
input_path = '../data/Integrated.pkl'

if os.path.exists(input_path):
    df = pd.read_pickle(input_path)
    print(f"✅ Dataset cargado correctamente.")
    print(f"📊 Registros totales: {df.shape[0]} | Columnas: {df.shape[1]}")
else:
    print(f"❌ Error: No se encontró el archivo en {input_path}")

# Vista previa de las columnas que usaremos para Feature Engineering


✅ Dataset cargado correctamente.
📊 Registros totales: 119143 | Columnas: 42


In [2]:
display(df.head())
df.info()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,customer_state_num,seller_state_num,customer_state_num_pred,seller_state_num_pred
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,25,25,25,25
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,25,25,25,25
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,25,25,25,25
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50,4,25,4,25
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,e73b67b67587f7644d5bd1a52deb1b01,5.0,NaN,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58,8,25,8,25


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119143 entries, 0 to 119142
Data columns (total 42 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       119143 non-null  object 
 1   customer_id                    119143 non-null  object 
 2   order_status                   119143 non-null  object 
 3   order_purchase_timestamp       119143 non-null  object 
 4   order_approved_at              118966 non-null  object 
 5   order_delivered_carrier_date   117057 non-null  object 
 6   order_delivered_customer_date  115722 non-null  object 
 7   order_estimated_delivery_date  119143 non-null  object 
 8   customer_unique_id             119143 non-null  object 
 9   customer_zip_code_prefix       119143 non-null  int64  
 10  customer_city                  119143 non-null  object 
 11  order_item_id                  118310 non-null  float64
 12  product_id                    

In [3]:
print("Iniciando Feature Engineering base...")

# 1. CONVERSIÓN DE FECHAS (De texto a objetos Datetime de Pandas)
# Esto es vital para poder hacer restas matemáticas entre fechas
date_columns = [
    'order_purchase_timestamp', 
    'order_estimated_delivery_date', 
    'order_delivered_customer_date'
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')



Iniciando Feature Engineering base...


In [4]:
# 2. CREACIÓN DE LA VARIABLE OBJETIVO (Target: is_delayed)
# Es 1 si la fecha real de entrega es MAYOR que la fecha estimada.
# Si el pedido aún no se ha entregado (NaT), por ahora lo dejamos como NaN para decidir luego.
df['is_delayed'] = np.where(
    df['order_delivered_customer_date'].notna(),
    (df['order_delivered_customer_date'] > df['order_estimated_delivery_date']).astype(int),
    np.nan
)



In [9]:
# 3. CREACIÓN DE NUEVAS VARIABLES (Feature Engineering)

# A. Factor Temporal: ¿Cuántos días prometió Olist que tardaría?
df['estimated_delivery_margin_days'] = (df['order_estimated_delivery_date'] - df['order_purchase_timestamp']).dt.days

# B. Factor Temporal: Estacionalidad (Mes y Día de la semana de la compra)
df['purchase_month'] = df['order_purchase_timestamp'].dt.month
df['purchase_day_of_week'] = df['order_purchase_timestamp'].dt.dayofweek # 0=Lunes, 6=Domingo

# C. Factor Físico: Volumen del paquete en cm cúbicos
df['product_volume_cm3'] = df['product_length_cm'] * df['product_height_cm'] * df['product_width_cm']

# D. Factor Geográfico: ¿Es un envío dentro del mismo estado? 
# Usamos las columnas que inferimos con tu red neuronal
df['is_same_state'] = (df['customer_state_num_pred'] == df['seller_state_num_pred']).astype(int)

# E. Tiempo Real: ¿Cuánto tardó realmente en llegar?
# Nota: Esta columna es solo para análisis, NO para alimentar la IA (Data Leakage)
df['actual_delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days



In [6]:
# 4. SELECCIÓN ESTRICTA DE COLUMNAS (Eliminando Data Leakage y ruido)
# Solo nos quedamos con lo que sabemos en el momento en que el cliente hace clic en "Comprar"
features_to_keep = [
    # Identificadores (útiles para trazabilidad, se quitarán antes de entrenar)
    'order_id', 
    
    # Target
    'is_delayed', # Para clasificación
    'actual_delivery_days', # Para regresión
    
    # Variables Temporales (Conocidas al comprar)
    'estimated_delivery_margin_days',
    'purchase_month',
    'purchase_day_of_week',
    
    # Variables Físicas / Producto
    'product_weight_g',
    'product_volume_cm3',
    'product_category_name_english', # Usamos la versión en inglés si existe
    
    # Variables Geográficas
    'customer_zip_code_prefix',
    'seller_zip_code_prefix',
    'is_same_state',
    'customer_state_num_pred', # Mantenemos los IDs de tu IA
    'seller_state_num_pred',
    
    # Variables de Transacción
    'freight_value',
    'price'
]

# Creamos nuestro dataset final de modelado
df_ml = df[features_to_keep].copy()

print("✅ Feature Engineering completado.")
print(f"Dimensiones del dataset para ML: {df_ml.shape}")

# Veamos cuántos retrasos tenemos actualmente (sin contar los nulos)
print("\nDistribución de la variable objetivo (is_delayed):")
print(df_ml['is_delayed'].value_counts(dropna=False))

display(df_ml.head())

✅ Feature Engineering completado.
Dimensiones del dataset para ML: (119143, 16)

Distribución de la variable objetivo (is_delayed):
is_delayed
0.0    106654
1.0      9068
NaN      3421
Name: count, dtype: int64


,order_id,is_delayed,actual_delivery_days,estimated_delivery_margin_days,purchase_month,purchase_day_of_week,product_weight_g,product_volume_cm3,product_category_name_english,customer_zip_code_prefix,seller_zip_code_prefix,is_same_state,customer_state_num_pred,seller_state_num_pred,freight_value,price
0,e481f51cbdc54678b7cc49136f2d6af7,0.0,8.0,8.0,10,0,500.0,1976.0,housewares,3149,9350.0,1,25,25,8.72,29.99
1,e481f51cbdc54678b7cc49136f2d6af7,0.0,8.0,8.0,10,0,500.0,1976.0,housewares,3149,9350.0,1,25,25,8.72,29.99
2,e481f51cbdc54678b7cc49136f2d6af7,0.0,8.0,8.0,10,0,500.0,1976.0,housewares,3149,9350.0,1,25,25,8.72,29.99
3,53cdb2fc8bc7dce0b6741e2150273451,0.0,13.0,13.0,7,1,400.0,4693.0,perfumery,47813,31570.0,0,4,25,22.76,118.70
4,47770eb9100c2d0c44946d9cf07ec65d,0.0,9.0,9.0,8,2,420.0,9576.0,auto,75265,14840.0,0,8,25,19.22,159.90


In [7]:
display(df_ml.head())
df_ml.info()

,order_id,is_delayed,actual_delivery_days,estimated_delivery_margin_days,purchase_month,purchase_day_of_week,product_weight_g,product_volume_cm3,product_category_name_english,customer_zip_code_prefix,seller_zip_code_prefix,is_same_state,customer_state_num_pred,seller_state_num_pred,freight_value,price
0,e481f51cbdc54678b7cc49136f2d6af7,0.0,8.0,8.0,10,0,500.0,1976.0,housewares,3149,9350.0,1,25,25,8.72,29.99
1,e481f51cbdc54678b7cc49136f2d6af7,0.0,8.0,8.0,10,0,500.0,1976.0,housewares,3149,9350.0,1,25,25,8.72,29.99
2,e481f51cbdc54678b7cc49136f2d6af7,0.0,8.0,8.0,10,0,500.0,1976.0,housewares,3149,9350.0,1,25,25,8.72,29.99
3,53cdb2fc8bc7dce0b6741e2150273451,0.0,13.0,13.0,7,1,400.0,4693.0,perfumery,47813,31570.0,0,4,25,22.76,118.70
4,47770eb9100c2d0c44946d9cf07ec65d,0.0,9.0,9.0,8,2,420.0,9576.0,auto,75265,14840.0,0,8,25,19.22,159.90


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119143 entries, 0 to 119142
Data columns (total 16 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   order_id                        119143 non-null  object 
 1   is_delayed                      115722 non-null  float64
 2   actual_delivery_days            115722 non-null  float64
 3   estimated_delivery_margin_days  115722 non-null  float64
 4   purchase_month                  119143 non-null  int32  
 5   purchase_day_of_week            119143 non-null  int32  
 6   product_weight_g                118290 non-null  float64
 7   product_volume_cm3              118290 non-null  float64
 8   product_category_name_english   116576 non-null  object 
 9   customer_zip_code_prefix        119143 non-null  int64  
 10  seller_zip_code_prefix          118310 non-null  float64
 11  is_same_state                   119143 non-null  int64  
 12  customer_state_n

In [8]:
import os

# Asegurarnos de que la carpeta existe (por si acaso)
os.makedirs('../data', exist_ok=True)

# Definir las rutas (he corregido el nombre a 'Engineered', cámbialo si el error era intencional)
ruta_pkl = '../data/Engineered.pkl'

# 2. Guardar como Pickle (Opcional, pero muy recomendado para no perder los tipos de datos 'int' de tus IDs)
df_ml.to_pickle(ruta_pkl)
print(f"✅ Dataset maestro guardado en formato Pickle: {ruta_pkl}")

# Comprobación de tamaño
tamano_mb = os.path.getsize(ruta_pkl) / (1024 * 1024)
print(f"📦 Tamaño del archivo Pickle: {tamano_mb:.2f} MB")

✅ Dataset maestro guardado en formato Pickle: ../data/Engineered.pkl
📦 Tamaño del archivo Pickle: 15.82 MB
